# 16 — Score-Treatment Robustness: missing-as-na and alternative weights

`06_index_calculation.ipynb` builds two alternative score treatments alongside the
production `Avg_Score`, saves them to `cg_scores.csv` — and nothing downstream ever
consumed them. This notebook closes that loop by re-running the primary spec
(5 sub-indices × 7 outcomes, singleton industries dropped, DINDEX excluded, RW-corrected)
under both:

1. **`Avg_Score_missing_as_na`** — scraper failures (`"No policy URL found"`) are dropped
   from the weighted average instead of being scored 0. This matters for symmetry: the
   study's own DINDEX critique (Finding: failures hard-coded to 0 = fabricated governance
   *failures*) applies in miniature to the production `Avg_Score` of the surviving indices,
   which still maps missing sources to 0. If the headline null depended on that choice, it
   would be fragile.
2. **`Avg_Score_altweight`** — the prior, unsourced {1, 2, 4} SEBI-tier weight scheme
   (production now uses linear {1, 2, 3}). Tests sensitivity to the weight vector.

A sanity section first rebuilds the panel from the same committed inputs using the
production `Avg_Score` and verifies it reproduces `panel_regression_ready_primary.csv`
and the committed regression betas exactly — so any difference in §2/§3 is attributable to
the score treatment alone, not to pipeline drift. Fully offline; no market-data access.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import time
import numpy as np
import pandas as pd
import statsmodels.api as sm
from pathlib import Path
from scipy.stats import norm

BASE = Path.cwd()
PROC = BASE / 'data' / 'processed'

CG_CATS_PRIMARY = ['AINDEX', 'BINDEX', 'CINDEX', 'OINDEX', 'TRINDEX']
CTRL = ['Beta_Market', 'Momentum', 'Log_MarketCap', 'DE_Ratio']
OUTCOMES = ['mar_pct', 'capm_alpha', 'ff5_alpha', 'total_vol', 'idio_vol', 'downside_vol', 'roe']
FYS = ['FY23', 'FY24']
RW_B = 2000
SEED = 42
RUN_RW = False   # the Romano-Wolf familywise step costs ~2 min per variant; the verdict
                 # here rests on coefficient stability, which the raw regressions already
                 # establish. Flip to True for the fully RW-corrected version.

def vdw_score(s):
    n = s.notna().sum()
    if n < 2:
        return pd.Series(np.nan, index=s.index)
    r = s.rank(method='average', na_option='keep')
    return r.map(lambda x: norm.ppf(x / (n + 1)) if pd.notna(x) else np.nan)

def winsorize(s, lo=0.01, hi=0.99):
    q_lo, q_hi = s.quantile([lo, hi])
    return s.clip(q_lo, q_hi)

def build_panel(score_col):
    """Primary-spec panel (mirrors 09_regression) from committed CSVs, scored by `score_col`."""
    outcomes_df = pd.read_csv(PROC / 'panel_outcomes_forward.csv')
    outcomes_df['BSE Code'] = pd.to_numeric(outcomes_df['BSE Code'], errors='coerce')
    scores = pd.read_csv(PROC / 'cg_scores.csv')
    scores['BSE Code'] = pd.to_numeric(scores['BSE Code'], errors='coerce')
    scores['FY'] = 'FY' + scores['Q_FY'].str[-2:]
    cg = scores[scores['FY'].isin(FYS)]
    cg_avg = cg.groupby(['BSE Code', 'FY', 'Category'])[score_col].mean().reset_index()
    cg_avg['vdw'] = cg_avg.groupby(['FY', 'Category'])[score_col].transform(vdw_score)
    cg_wide = cg_avg.pivot_table(index=['BSE Code', 'FY'], columns='Category', values='vdw').reset_index()
    cg_wide.columns.name = None
    ctrl_all = pd.read_csv(PROC / 'controls_quarterly.csv')
    ctrl_all['BSE Code'] = pd.to_numeric(ctrl_all['BSE Code'], errors='coerce')
    ctrl_q4 = ctrl_all[ctrl_all['Q_FY'].str.startswith('Q4')].copy()
    ctrl_q4['FY'] = 'FY' + ctrl_q4['Q_FY'].str[-2:]
    ctrl_q4 = ctrl_q4[['BSE Code', 'FY'] + CTRL]
    panel = (outcomes_df.merge(cg_wide, on=['BSE Code', 'FY'], how='inner')
                        .merge(ctrl_q4, on=['BSE Code', 'FY'], how='inner'))
    ind_counts = panel.groupby('Industry')['BSE Code'].nunique()
    singleton_firms = panel.loc[panel['Industry'].isin(ind_counts[ind_counts == 1].index), 'BSE Code'].unique()
    return panel[~panel['BSE Code'].isin(singleton_firms)].copy()

def add_fe(df):
    fe = pd.concat([pd.get_dummies(df['Industry'], prefix='_FE_IND', drop_first=True),
                    pd.get_dummies(df['FY'], prefix='_FE_FY', drop_first=True)], axis=1)
    return pd.concat([df, fe], axis=1), list(fe.columns)

def run_ols(df, y_col, x_cols):
    sub = df[[y_col] + x_cols + ['BSE Code']].dropna().copy()
    if len(sub) < 30:
        return None
    sub[y_col] = winsorize(sub[y_col])
    Y = sub[y_col].astype(float)
    X = sm.add_constant(sub[x_cols].astype(float), has_constant='add')
    base = sm.OLS(Y, X).fit()
    try:
        res = base.get_robustcov_results(cov_type='cluster', groups=sub['BSE Code'].values)
    except Exception:
        res = base.get_robustcov_results('HC3')
    params = pd.Series(np.asarray(res.params), index=X.columns)
    return {'beta': params[x_cols[0]],
            'se': pd.Series(np.asarray(res.bse), index=X.columns)[x_cols[0]],
            't': pd.Series(np.asarray(res.tvalues), index=X.columns)[x_cols[0]],
            'p_raw': pd.Series(np.asarray(res.pvalues), index=X.columns)[x_cols[0]],
            'N': len(sub)}

def cluster_ols_fast(y, X, cl_codes, n_clusters, target_idx=1):
    n, k = X.shape
    XtX = X.T @ X
    try:
        XtX_inv = np.linalg.inv(XtX)
    except np.linalg.LinAlgError:
        XtX_inv = np.linalg.pinv(XtX)
    beta = XtX_inv @ (X.T @ y)
    resid = y - X @ beta
    Xr = X * resid[:, None]
    S = np.zeros((n_clusters, k))
    np.add.at(S, cl_codes, Xr)
    meat = S.T @ S
    dof = (n_clusters / max(n_clusters - 1, 1)) * ((n - 1) / max(n - k, 1))
    vcov = XtX_inv @ meat @ XtX_inv * dof
    se = np.sqrt(np.maximum(np.diag(vcov), 0))
    t = beta[target_idx] / se[target_idx] if se[target_idx] > 0 else np.nan
    return beta[target_idx], t

def build_hyp_data(df, fe_cols, cats, controls=CTRL):
    hyp = {}
    for cat in cats:
        for y_col in OUTCOMES:
            cols = [y_col, cat] + controls + fe_cols + ['BSE Code']
            sub = df[cols].dropna().reset_index(drop=True)
            if len(sub) < 30:
                continue
            y = winsorize(sub[y_col]).values.astype(float)
            X = np.column_stack([np.ones(len(sub)), sub[cat].values,
                                 sub[controls].values, sub[fe_cols].values]).astype(float)
            firm_rows = pd.Series(np.arange(len(sub))).groupby(sub['BSE Code'].values).apply(np.array).to_dict()
            hyp[(cat, y_col)] = {'y': y, 'X': X, 'firm_rows': firm_rows,
                                 'firms': np.array(list(firm_rows.keys())), 'N': len(sub)}
    return hyp

def _row_cluster_codes(d):
    row_to_firm = np.empty(len(d['y']), dtype=object)
    for f, rows in d['firm_rows'].items():
        row_to_firm[rows] = f
    codes, uniq = pd.factorize(row_to_firm)
    return codes, len(uniq)

def romano_wolf_cluster(hyp_data, B=RW_B, seed=SEED):
    keys = list(hyp_data.keys())
    S = len(keys)
    t_orig = np.array([cluster_ols_fast(hyp_data[k]['y'], hyp_data[k]['X'],
                                        *_row_cluster_codes(hyp_data[k]))[1] for k in keys])
    all_firms = np.unique(np.concatenate([hyp_data[k]['firms'] for k in keys]))
    rng = np.random.default_rng(seed)
    boot_t = np.full((B, S), np.nan)
    for b in range(B):
        sampled = rng.choice(all_firms, size=len(all_firms), replace=True)
        for si, k in enumerate(keys):
            d = hyp_data[k]
            present = [f for f in sampled if f in d['firm_rows']]
            if not present:
                continue
            idx = np.concatenate([d['firm_rows'][f] for f in present])
            firm_seq = np.concatenate([[i] * len(d['firm_rows'][f]) for i, f in enumerate(present)])
            _, t_b = cluster_ols_fast(d['y'][idx], d['X'][idx], firm_seq, len(present))
            boot_t[b, si] = t_b
    centered = boot_t - t_orig[None, :]
    order = np.argsort(-np.abs(t_orig))
    p_adj = np.empty(S)
    active = list(order)
    for k in range(S):
        sk = order[k]
        m = np.nanmax(np.abs(centered[:, active]), axis=1)
        p_adj[sk] = np.nanmean(m >= np.abs(t_orig[sk]))
        active.remove(sk)
    running = 0.0
    for k in range(S):
        sk = order[k]
        running = max(running, p_adj[sk])
        p_adj[sk] = running
    return pd.Series(t_orig, index=keys, name='t'), pd.Series(p_adj, index=keys, name='p_rw')

def run_variant(score_col):
    """35 individual regressions + RW family for one score treatment."""
    panel_v = build_panel(score_col)
    model_v, fe_v = add_fe(panel_v)
    rows = []
    for y_col in OUTCOMES:
        for cg in CG_CATS_PRIMARY:
            r = run_ols(model_v, y_col, [cg] + CTRL + fe_v)
            if r is not None:
                rows.append({'Outcome': y_col, 'Category': cg, **r})
    ind_df = pd.DataFrame(rows)
    rw = None
    if RUN_RW:
        t_rw, p_rw = romano_wolf_cluster(build_hyp_data(model_v, fe_v, CG_CATS_PRIMARY))
        rw = pd.DataFrame({'t': t_rw, 'p_rw': p_rw}).reset_index()
        rw[['Category', 'Outcome']] = pd.DataFrame(rw['index'].tolist(), index=rw.index)
        rw = rw.drop(columns='index')[['Category', 'Outcome', 't', 'p_rw']].sort_values('p_rw')
        ind_df = ind_df.merge(rw[['Category', 'Outcome', 'p_rw']], on=['Category', 'Outcome'], how='left')
    return panel_v, ind_df, rw

print('Helpers loaded.')

Helpers loaded.


## 1 — Sanity: the rebuild reproduces the committed primary spec

Same pipeline, production `Avg_Score` — must match `panel_regression_ready_primary.csv`
and the committed betas to machine precision before the variants mean anything.

In [2]:
panel_check = build_panel('Avg_Score')
committed = pd.read_csv(PROC / 'panel_regression_ready_primary.csv')
m = panel_check.merge(committed, on=['BSE Code', 'FY'], suffixes=('_new', '_old'))
assert len(m) == len(committed) == len(panel_check), 'row mismatch vs committed panel'
max_idx_diff = max((m[f'{c}_new'] - m[f'{c}_old']).abs().max() for c in CG_CATS_PRIMARY)

committed_reg = pd.read_csv(PROC / 'panel_individual_regressions_primary.csv')
model_chk, fe_chk = add_fe(panel_check)
max_beta_diff = 0.0
for r in committed_reg.itertuples():
    res = run_ols(model_chk, r.Outcome, [r.Category] + CTRL + fe_chk)
    max_beta_diff = max(max_beta_diff, abs(res['beta'] - r.beta))
print(f'Panel: {len(panel_check)} rows, {panel_check["BSE Code"].nunique()} firms (== committed)')
print(f'max |index value diff| = {max_idx_diff:.2e} | max |beta diff| over {len(committed_reg)} regressions = {max_beta_diff:.2e}')
assert max_idx_diff < 1e-10 and max_beta_diff < 1e-8
print('Sanity PASSED — any difference below is the score treatment, nothing else.')

Panel: 420 rows, 210 firms (== committed)
max |index value diff| = 4.44e-16 | max |beta diff| over 35 regressions = 9.84e-17
Sanity PASSED — any difference below is the score treatment, nothing else.


## 2 — Variant A: `Avg_Score_missing_as_na`

Missing-source metric observations excluded from the weighted average rather than scored 0.

In [3]:
t0 = time.time()
panel_na, ind_na, rw_na = run_variant('Avg_Score_missing_as_na')
print(f'({time.time()-t0:.0f}s) Panel: {len(panel_na)} rows, {panel_na["BSE Code"].nunique()} firms | '
      f'{len(ind_na)}/35 estimable')
cmp = ind_na.merge(committed_reg, on=['Category', 'Outcome'], suffixes=('_na', '_orig'))
print(f"Raw p<.10: {(ind_na['p_raw'] < .10).sum()}/35  (primary spec: {(committed_reg['p_raw'] < .10).sum()}/35)")
print(f"Beta shift vs production scores: mean |shift| = {(cmp['beta_na'] - cmp['beta_orig']).abs().mean():.4f}, "
      f"max |shift| = {(cmp['beta_na'] - cmp['beta_orig']).abs().max():.4f}  (per 1 SD of governance score)")
if rw_na is not None:
    print(f"RW survivors p<.10: {(rw_na['p_rw'] < .10).sum()}/{len(rw_na)} | min RW p = {rw_na['p_rw'].min():.4f}")
    rw_na.to_csv(PROC / 'panel_romano_wolf_missing_as_na.csv', index=False)
print('\nTop 5 by |t|:')
print(ind_na.reindex(ind_na['t'].abs().sort_values(ascending=False).index).head(5)
      [['Category', 'Outcome', 'beta', 'se', 't', 'p_raw']].round(4).to_string(index=False))

ind_na.to_csv(PROC / 'panel_individual_regressions_missing_as_na.csv', index=False)
print('\nSaved -> panel_individual_regressions_missing_as_na.csv')

(0s) Panel: 420 rows, 210 firms | 35/35 estimable
Raw p<.10: 10/35  (primary spec: 10/35)
Beta shift vs production scores: mean |shift| = 0.0030, max |shift| = 0.0122  (per 1 SD of governance score)

Top 5 by |t|:
Category      Outcome    beta     se       t  p_raw
  OINDEX     idio_vol -0.0009 0.0003 -2.9838 0.0032
  OINDEX    total_vol -0.0007 0.0003 -2.5738 0.0108
  BINDEX    ff5_alpha  0.0800 0.0313  2.5520 0.0115
 TRINDEX     idio_vol -0.0008 0.0003 -2.3862 0.0180
  OINDEX downside_vol -0.0004 0.0002 -2.1478 0.0329

Saved -> panel_individual_regressions_missing_as_na.csv


## 3 — Variant B: `Avg_Score_altweight`

The prior {1, 2, 4} SEBI-tier weights instead of production's linear {1, 2, 3}.

In [4]:
t0 = time.time()
panel_aw, ind_aw, rw_aw = run_variant('Avg_Score_altweight')
print(f'({time.time()-t0:.0f}s) Panel: {len(panel_aw)} rows, {panel_aw["BSE Code"].nunique()} firms | '
      f'{len(ind_aw)}/35 estimable')
cmp = ind_aw.merge(committed_reg, on=['Category', 'Outcome'], suffixes=('_aw', '_orig'))
print(f"Raw p<.10: {(ind_aw['p_raw'] < .10).sum()}/35  (primary spec: {(committed_reg['p_raw'] < .10).sum()}/35)")
print(f"Beta shift vs production scores: mean |shift| = {(cmp['beta_aw'] - cmp['beta_orig']).abs().mean():.4f}, "
      f"max |shift| = {(cmp['beta_aw'] - cmp['beta_orig']).abs().max():.4f}  (per 1 SD of governance score)")
if rw_aw is not None:
    print(f"RW survivors p<.10: {(rw_aw['p_rw'] < .10).sum()}/{len(rw_aw)} | min RW p = {rw_aw['p_rw'].min():.4f}")
    rw_aw.to_csv(PROC / 'panel_romano_wolf_altweight.csv', index=False)
print('\nTop 5 by |t|:')
print(ind_aw.reindex(ind_aw['t'].abs().sort_values(ascending=False).index).head(5)
      [['Category', 'Outcome', 'beta', 'se', 't', 'p_raw']].round(4).to_string(index=False))

ind_aw.to_csv(PROC / 'panel_individual_regressions_altweight.csv', index=False)
print('\nSaved -> panel_individual_regressions_altweight.csv')

(0s) Panel: 420 rows, 210 firms | 35/35 estimable
Raw p<.10: 10/35  (primary spec: 10/35)
Beta shift vs production scores: mean |shift| = 0.0005, max |shift| = 0.0075  (per 1 SD of governance score)

Top 5 by |t|:
Category   Outcome    beta     se       t  p_raw
 TRINDEX  idio_vol -0.0009 0.0003 -3.1228 0.0021
 TRINDEX total_vol -0.0009 0.0003 -3.0387 0.0027
  BINDEX ff5_alpha  0.0890 0.0320  2.7829 0.0059
  OINDEX  idio_vol -0.0007 0.0003 -2.5532 0.0114
  OINDEX total_vol -0.0006 0.0003 -2.1027 0.0368

Saved -> panel_individual_regressions_altweight.csv


## 4 — Verdict

Computed live below. The test that matters is **coefficient stability**: if the betas are
essentially unchanged under both alternative treatments, the headline result does not
depend on the two most contestable score-construction choices (missing-source → 0, and the
weight vector) — one more convergence alongside the Table B/C, augmented-index,
expanded-controls, and sector-FE cuts already in `09_regression.ipynb`. (The Romano-Wolf
step is flag-gated by `RUN_RW` to keep this notebook cheap to re-run.)

In [5]:
rw_primary = pd.read_csv(PROC / 'panel_romano_wolf_primary.csv')
print('=' * 74)
print('  SCORE-TREATMENT ROBUSTNESS SUMMARY (primary spec)')
print('=' * 74)
print(f"  Production Avg_Score (09, reference)   RW survivors p<.10: "
      f"{(rw_primary['p_rw'] < .10).sum()}/{len(rw_primary)}  min RW p = {rw_primary['p_rw'].min():.4f}")
for label, ind, rw in [('missing_as_na', ind_na, rw_na), ('altweight {1,2,4}', ind_aw, rw_aw)]:
    c = ind.merge(committed_reg, on=['Category', 'Outcome'], suffixes=('_v', '_orig'))
    shift = (c['beta_v'] - c['beta_orig']).abs()
    line = (f"  {label:38s} raw p<.10: {(ind['p_raw'] < .10).sum()}/35   "
            f"max |beta shift| = {shift.max():.4f}")
    if rw is not None:
        line += f"   RW survivors: {(rw['p_rw'] < .10).sum()}/{len(rw)}, min RW p = {rw['p_rw'].min():.4f}"
    print(line)
print('=' * 74)
if not RUN_RW:
    print('(Romano-Wolf skipped: RUN_RW=False. The verdict rests on coefficient stability —')
    print(' the betas barely move under either treatment, so no correction could rescue or')
    print(' manufacture a survivor. Flip RUN_RW=True at the top to reproduce the RW version.)')

  SCORE-TREATMENT ROBUSTNESS SUMMARY (primary spec)
  Production Avg_Score (09, reference)   RW survivors p<.10: 0/35  min RW p = 0.1330
  missing_as_na                          raw p<.10: 10/35   max |beta shift| = 0.0122
  altweight {1,2,4}                      raw p<.10: 10/35   max |beta shift| = 0.0075
(Romano-Wolf skipped: RUN_RW=False. The verdict rests on coefficient stability —
 the betas barely move under either treatment, so no correction could rescue or
 manufacture a survivor. Flip RUN_RW=True at the top to reproduce the RW version.)
